In [1]:
print("Test")

Test


In [2]:
# Imports
import sys
import pathlib

# Add the project's root directory to the Python path
sys.path.append(pathlib.Path("../").resolve().as_posix())

# Configurations
seed = 42

# Paths
DATA_DIR = pathlib.Path("../data/")
ENC2017_ROOT = DATA_DIR / "enc2017"
UD_ET_EDT_ROOT = DATA_DIR / "ud_et_edt"
HOMONYMS_ROOT = DATA_DIR / "homonymous_word_forms"

ENC2017_DIRS = {
    "processed": ENC2017_ROOT / "processed",
    "raw": ENC2017_ROOT / "raw",
}

UD_ET_EDT_DIRS = {
    "processed": UD_ET_EDT_ROOT / "processed",
    "raw": UD_ET_EDT_ROOT / "raw",
}

HOMONYMS_DIRS = {
    "processed": HOMONYMS_ROOT / "processed",
    "annotations": HOMONYMS_ROOT / "annotations",
}

OUTPUT_DIR = pathlib.Path("../outputs/")

MODEL_DIR = pathlib.Path("../models/")

In [3]:
import pandas as pd
import numpy as np
import ast
import json
import tqdm
import estnltk
from estnltk.default_resolver import make_resolver
from estnltk.taggers import VabamorfAnalyzer

from typing import Any

import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer, pipeline
from scripts.model.bert_morph_tagger import BertMorphTagger

from datasets import Dataset
from transformers.pipelines.pt_utils import KeyDataset

e:\Git_projects\EstNLTK\simpletransformers\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Morphological tagging using BERT MLM predictions


In [4]:
homonyms_df = pd.read_parquet(
    HOMONYMS_DIRS["processed"] / "homonyms_overall_sentences.parquet"
)
hom_df = pd.read_parquet(HOMONYMS_DIRS["processed"] / "homonyms_overall.parquet")
display(homonyms_df.head())
display(hom_df.head())

,num,inflection_type,sentence,word,word_span,label,source
0,1,1,"Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.",võitja,"[74, 80]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
1,1,1,"Normi-aktiveerimise teooria (Schwartz, 1970) on algselt mõeldud moraalse otsustamisprotsessi analüüsimiseks abistava käitumise näitel.",teooria,"[20, 27]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
2,1,1,"""Ehk oleks mõttekas ka mõni selleteemaline hoiatav kampaania korraldada,"" lisab punase autoga preili.",kampaania,"[51, 60]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
3,1,1,"""Minu otsus oli õige ning teeksin kõik sama moodi, kui saaksin uuesti teha,"" kommenteerib kolm aastat tagasi eriliste teenete eest Eesti passi saanud Primakov.",õige,"[16, 20]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
4,1,1,"Itaalia president ütles Venemaa riigipea auks korraldatud suurejoonelisel banketil, et kahe riigi ühisavaldus Iraagi kohta oli kahe riigipea ""suur tarkuseavaldus"".",Itaalia,"[0, 7]",[sg g],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json


,sentence_id,words,form,pos,labels,infl_type,source
0,0,Edinburghi,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
1,0,agulite,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
2,0,mehe,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
3,0,Irvine,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
4,0,Welshi,-,-,-,1,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json


In [5]:
# Extract pos info from hom_df and add it to homonyms_df
assert len(homonyms_df) == len(hom_df[hom_df["pos"] != "-"])
homonyms_df["pos"] = hom_df[hom_df["pos"] != "-"]["pos"].values
display(homonyms_df.head())
# Move column "pos" to the sixth position (after "label")
cols = homonyms_df.columns.tolist()
cols.insert(6, cols.pop(cols.index("pos")))
homonyms_df = homonyms_df[cols]
display(homonyms_df.head())

,num,inflection_type,sentence,word,word_span,label,source,pos
0,1,1,"Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.",võitja,"[74, 80]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json,S
1,1,1,"Normi-aktiveerimise teooria (Schwartz, 1970) on algselt mõeldud moraalse otsustamisprotsessi analüüsimiseks abistava käitumise näitel.",teooria,"[20, 27]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json,S
2,1,1,"""Ehk oleks mõttekas ka mõni selleteemaline hoiatav kampaania korraldada,"" lisab punase autoga preili.",kampaania,"[51, 60]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json,S
3,1,1,"""Minu otsus oli õige ning teeksin kõik sama moodi, kui saaksin uuesti teha,"" kommenteerib kolm aastat tagasi eriliste teenete eest Eesti passi saanud Primakov.",õige,"[16, 20]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json,A
4,1,1,"Itaalia president ütles Venemaa riigipea auks korraldatud suurejoonelisel banketil, et kahe riigi ühisavaldus Iraagi kohta oli kahe riigipea ""suur tarkuseavaldus"".",Itaalia,"[0, 7]",[sg g],infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json,H


,num,inflection_type,sentence,word,word_span,label,pos,source
0,1,1,"Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.",võitja,"[74, 80]",[sg n],S,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
1,1,1,"Normi-aktiveerimise teooria (Schwartz, 1970) on algselt mõeldud moraalse otsustamisprotsessi analüüsimiseks abistava käitumise näitel.",teooria,"[20, 27]",[sg n],S,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
2,1,1,"""Ehk oleks mõttekas ka mõni selleteemaline hoiatav kampaania korraldada,"" lisab punase autoga preili.",kampaania,"[51, 60]",[sg n],S,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
3,1,1,"""Minu otsus oli õige ning teeksin kõik sama moodi, kui saaksin uuesti teha,"" kommenteerib kolm aastat tagasi eriliste teenete eest Eesti passi saanud Primakov.",õige,"[16, 20]",[sg n],A,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json
4,1,1,"Itaalia president ütles Venemaa riigipea auks korraldatud suurejoonelisel banketil, et kahe riigi ühisavaldus Iraagi kohta oli kahe riigipea ""suur tarkuseavaldus"".",Itaalia,"[0, 7]",[sg g],H,infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json


In [6]:
print("Unique POS tags in the dataset:", homonyms_df["pos"].unique())

Unique POS tags in the dataset: ['S' 'A' 'H' 'P' 'N' 'V']


In [7]:
from collections.abc import Iterable
import time

# Model name or path
MODEL_NAME = "EMBEDDIA/est-roberta"

# Device setup: GPU when available, otherwise CPU.
DEVICE = 0 if torch.cuda.is_available() else -1
print(f"Using model: {MODEL_NAME}")
print("Using device:", "cuda" if DEVICE == 0 else "cpu")

# Initialise tokenizer and MLM model.
mlm_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
mlm_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

# High-level fill-mask pipeline for easy TOP-k extraction.
fill_mask = pipeline(
    task="fill-mask",
    model=mlm_model,
    tokenizer=mlm_tokenizer,
    device=DEVICE,
    framework="pt",
)

# BERT-style models use [MASK] as the masking token.
MASK_TOKEN = mlm_tokenizer.mask_token
print("Mask token:", MASK_TOKEN)


def prepare_mlm_input(text: str, masked_word_span: tuple[int, int]) -> str:
    """
    Prepare input text for masked language modeling by replacing the specified word span with a mask token.
    Args:
        text: The original input text.
        masked_word_span: A tuple (start_index, end_index) indicating the character span of the word to mask.
    Returns:
        A new string with the specified span replaced by the mask token.
    Raises:
        ValueError: If the masked_word_span is invalid (e.g., out of bounds or start >= end).
    """
    start, end = masked_word_span
    if start < 0 or end > len(text) or start >= end:
        raise ValueError(
            f"Invalid masked_word_span: {masked_word_span}. Must be within text bounds and start < end."
        )
    return text[:start] + MASK_TOKEN + text[end:]


def parse_span(span_value: object) -> tuple[int, int]:
    """Parse a span value into a `(start, end)` integer tuple.

    Args:
        span_value: Span in tuple form or string form like `"(74, 80)"`.

    Returns:
        Parsed `(start, end)` tuple.

    Raises:
        ValueError: If the input cannot be parsed into two integers.
    """
    if isinstance(span_value, tuple) and len(span_value) == 2:
        return (int(span_value[0]), int(span_value[1]))
    if isinstance(span_value, str):
        parsed = ast.literal_eval(span_value)
        if isinstance(parsed, tuple) and len(parsed) == 2:
            return (int(parsed[0]), int(parsed[1]))
    if isinstance(span_value, np.ndarray) and span_value.shape == (2,):
        return (int(span_value[0]), int(span_value[1]))
    raise ValueError(f"Invalid span value: {span_value}")


def parse_label_form(label_value: object) -> str:
    """Extract the true form label from dataset label column.

    Args:
        label_value: Label value in string-list form (e.g. `"['sg n']"`) or iterable.

    Returns:
        A single form label string (e.g. `"sg n"`).

    Raises:
        ValueError: If no form label can be extracted.
    """
    if isinstance(label_value, str):
        parsed = ast.literal_eval(label_value)
        if isinstance(parsed, list) and parsed:
            return str(parsed[0]).strip()
        return str(label_value).strip()
    if isinstance(label_value, Iterable):
        as_list = list(label_value)
        if as_list:
            return str(as_list[0]).strip()
    raise ValueError(f"Invalid label value: {label_value}")


def sentence_with_token_replacement(
    sentence: str,
    original_span: tuple[int, int],
    replacement_token: str,
) -> tuple[str, tuple[int, int]]:
    """Replace the original word span with a predicted token and return new span.

    Args:
        sentence: Original sentence.
        original_span: Character span `(start, end)` for the original token.
        replacement_token: Predicted token to inject.

    Returns:
        A tuple of `(candidate_sentence, candidate_span)`.
    """
    start, end = original_span
    candidate_sentence = sentence[:start] + replacement_token + sentence[end:]
    candidate_span = (start, start + len(replacement_token))
    return candidate_sentence, candidate_span


def extract_labels_at_span_with_resolver(
    sentence: str,
    span: tuple[int, int],
    resolver_obj: Any,
    text_cache: dict[str, estnltk.Text],
) -> list[str]:
    """Resolve morph analysis on full sentence and extract labels for a target span.

    Args:
        sentence: Candidate sentence to analyse.
        span: Target character span `(start, end)` in candidate sentence.
        resolver_obj: EstNLTK resolver.
        text_cache: Cache mapping sentence string to tagged EstNLTK Text object.

    Returns:
        Sorted list of unique form values for analyses matching the target span.
    """
    if sentence not in text_cache:
        text_obj = estnltk.Text(sentence)
        text_obj.tag_layer(resolver=resolver_obj)
        text_cache[sentence] = text_obj

    text_obj = text_cache[sentence]
    labels = list()
    target_start, target_end = span

    for analysis in text_obj.morph_analysis:
        if analysis.start == target_start and analysis.end == target_end:
            form_value = analysis["form"]
            pos_value = analysis["partofspeech"]
            if isinstance(form_value, str):
                labels.append((str(form_value).strip(), str(pos).strip()))
            else:
                for form, pos in zip(form_value, pos_value):
                    labels.append((str(form).strip(), str(pos).strip()))

    return sorted(label for label in labels if label)


bmt = BertMorphTagger("../models/NER_mudel_v2/")


def predict_form_label_from_masked_sentence(
    sentence: str,
    masked_word_span: tuple[int, int],
    n_candidates: int = 20,
    top_k_labels: int = 5,
    model: str = "Vabamorf",
    ambiguity_policy: str = "split",
    resolver_obj: Any | None = None,
    text_cache: dict[str, estnltk.Text] | None = None,
) -> dict[str, Any]:
    """Predict the most probable morphological form label from a masked sentence.

    The function converts MLM token predictions into form-label probabilities by:
    1) masking the target span,
    2) getting top-N MLM candidates with probabilities,
    3) analysing each candidate in full sentence context via resolver-based morph analysis,
    4) aggregating candidate probabilities by resolved form labels.

    Args:
        sentence: Original sentence containing the homonym.
        masked_word_span: Character span `(start, end)` of the homonym in `sentence`.
        n_candidates: Number of MLM candidate tokens to request.
        top_k_labels: Number of top labels to return in the summary.
        model: Name of the model to use: Vabamorf or BertMorphTagger. If "BertMorphTagger", the function needs an initialised `bmt` object in scope.
        ambiguity_policy: How to handle candidates with multiple resolved forms.
            - "split": split candidate probability equally across all resolved forms.
            - "first": assign full candidate probability to the first resolved form only.
        resolver_obj: Optional pre-created resolver; if None, a new resolver is created.
        text_cache: Optional cache for resolver-tagged `estnltk.Text` objects keyed by sentence.

    Returns:
        A dictionary containing:
        - `best_label`: Most probable form label (or None if no labels resolved).
        - `best_label_score`: Aggregated probability for `best_label` (or 0.0).
        - `top_labels`: List of top form labels with aggregated probabilities.
        - `label_scores`: Full label -> aggregated probability mapping.
        - `masked_sentence`: Sentence with mask token.
        - `candidate_details`: Per-candidate diagnostic details.
    """
    if ambiguity_policy not in {"split", "first"}:
        raise ValueError(
            "Invalid ambiguity_policy. Use 'split' or 'first'. "
            f"Received: {ambiguity_policy}"
        )

    if top_k_labels < 1:
        raise ValueError(f"top_k_labels must be >= 1. Received: {top_k_labels}")
    if n_candidates < 1:
        raise ValueError(f"n_candidates must be >= 1. Received: {n_candidates}")

    local_resolver = resolver_obj or make_resolver()
    local_cache = text_cache if text_cache is not None else {}

    def _extract_forms_at_span(
        candidate_sentence: str,
        candidate_span: tuple[int, int],
        model: str,
    ) -> list[str]:
        """Extract unique form labels at exact span from resolver-based analysis."""
        if candidate_sentence not in local_cache:
            text_obj = estnltk.Text(candidate_sentence)
            text_obj.tag_layer("sentences")
            if model == "Vabamorf":
                text_obj.tag_layer(resolver=local_resolver)
                local_cache[candidate_sentence] = text_obj
            elif model == "BertMorphTagger":
                bmt.tag(text_obj)
                local_cache[candidate_sentence] = text_obj

        text_obj = local_cache[candidate_sentence]
        target_start, target_end = candidate_span
        forms: list[str] = []
        if model == "Vabamorf":
            for analysis in text_obj.morph_analysis:
                if analysis.start == target_start and analysis.end == target_end:
                    form_value = analysis["form"]
                    if isinstance(form_value, str):
                        form_clean = form_value.strip()
                        if form_clean:
                            forms.append(form_clean)
                    else:
                        for form_item in form_value:
                            form_clean = str(form_item).strip()
                            if form_clean:
                                forms.append(form_clean)
        else:  # BertMorphTagger
            for analysis in text_obj.bert_morph_tagging:
                if analysis.start == target_start and analysis.end == target_end:
                    form_value = analysis["form"]
                    if isinstance(form_value, str):
                        form_clean = form_value.strip()
                        if form_clean:
                            forms.append(form_clean)
                    else:
                        for form_item in form_value:
                            form_clean = str(form_item).strip()
                            if form_clean:
                                forms.append(form_clean)

        return sorted(set(forms))

    original_span = parse_span(masked_word_span)
    masked_sentence = prepare_mlm_input(sentence, original_span)
    raw_predictions = fill_mask(masked_sentence, top_k=n_candidates)

    label_scores: dict[str, float] = {}
    candidate_details: list[dict[str, Any]] = []

    for rank, prediction in enumerate(raw_predictions, start=1):
        token = str(prediction.get("token_str", "")).strip()
        score = float(prediction.get("score", 0.0))

        candidate_sentence, candidate_span = sentence_with_token_replacement(
            sentence=sentence,
            original_span=original_span,
            replacement_token=token,
        )

        resolved_forms = _extract_forms_at_span(
            candidate_sentence, candidate_span, model
        )

        if resolved_forms:
            if ambiguity_policy == "split":
                share = score / len(resolved_forms)
                for form_label in resolved_forms:
                    label_scores[form_label] = label_scores.get(form_label, 0.0) + share
            else:
                first_label = resolved_forms[0]
                label_scores[first_label] = label_scores.get(first_label, 0.0) + score

        candidate_details.append(
            {
                "rank": rank,
                "token": token,
                "score": score,
                "sequence": prediction.get("sequence", ""),
                "candidate_sentence": candidate_sentence,
                "candidate_span": candidate_span,
                "resolved_forms": resolved_forms,
            }
        )

    sorted_labels = sorted(label_scores.items(), key=lambda x: x[1], reverse=True)
    top_labels = [
        {"label": label, "score": float(score)}
        for label, score in sorted_labels[:top_k_labels]
    ]

    if top_labels:
        best_label = top_labels[0]["label"]
        best_label_score = top_labels[0]["score"]
    else:
        best_label = None
        best_label_score = 0.0

    return {
        "best_label": best_label,
        "best_label_score": best_label_score,
        "top_labels": top_labels,
        "label_scores": {k: float(v) for k, v in label_scores.items()},
        "masked_sentence": masked_sentence,
        "candidate_details": candidate_details,
    }

Using model: EMBEDDIA/est-roberta
Using device: cuda


Device set to use cuda:0


Mask token: <mask>


In [8]:
# Example usage on a single row (optional quick check).
example_row = homonyms_df.iloc[0]
example_sentence = example_row["sentence"]
example_masked_word = example_row["word"]
example_word_span = example_row["word_span"]
print("Original sentence:", example_sentence)
print("Word span to mask:", example_word_span)
print("To be masked word:", example_masked_word)
masked_sentence = prepare_mlm_input(example_sentence, example_word_span)
print("Masked sentence:", masked_sentence)
example = predict_form_label_from_masked_sentence(
    sentence=str(example_sentence),
    masked_word_span=parse_span(example_word_span),
    n_candidates=100,
    top_k_labels=5,
    model="Vabamorf",
    ambiguity_policy="split",
)
print("Predicted best label:", example["best_label"])
print("Top label predictions:")
for label_info in example["top_labels"]:
    print(f"  Label: {label_info['label']}, Score: {label_info['score']:.4f}")

Original sentence: Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.
Word span to mask: [74 80]
To be masked word: võitja
Masked sentence: Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri <mask> James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.
Predicted best label: sg n
Top label predictions:
  Label: sg n, Score: 0.7222
  Label: sg g, Score: 0.1866
  Label: pl n, Score: 0.0317
  Label: nud, Score: 0.0315
  Label: sg tr, Score: 0.0015


In [9]:
# Example usage on a single row (optional quick check).
example_row = homonyms_df.iloc[0]
example_sentence = example_row["sentence"]
example_masked_word = example_row["word"]
example_word_span = example_row["word_span"]
print("Original sentence:", example_sentence)
print("Word span to mask:", example_word_span)
print("To be masked word:", example_masked_word)
masked_sentence = prepare_mlm_input(example_sentence, example_word_span)
print("Masked sentence:", masked_sentence)
example = predict_form_label_from_masked_sentence(
    sentence=str(example_sentence),
    masked_word_span=parse_span(example_word_span),
    n_candidates=100,
    top_k_labels=5,
    model="BertMorphTagger",
    ambiguity_policy="split",
)
print("Predicted best label:", example["best_label"])
print("Top label predictions:")
for label_info in example["top_labels"]:
    print(f"  Label: {label_info['label']}, Score: {label_info['score']:.4f}")

Original sentence: Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.
Word span to mask: [74 80]
To be masked word: võitja
Masked sentence: Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri <mask> James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.
Predicted best label: sg n
Top label predictions:
  Label: sg n, Score: 0.7094
  Label: sg g, Score: 0.1689
  Label: sg tr, Score: 0.0015
  Label: s, Score: 0.0002
  Label: pl n, Score: 0.0002


In [ ]:
example_sentence = "Ma näen maja."
example_masked_word = "maja"
example_word_span = (
    example_sentence.index(example_masked_word),
    example_sentence.index(example_masked_word) + len(example_masked_word),
)
print("Original sentence:", example_sentence)
print("Word span to mask:", example_word_span)
print("To be masked word:", example_masked_word)
masked_sentence = prepare_mlm_input(example_sentence, example_word_span)
print("Masked sentence:", masked_sentence)
example = predict_form_label_from_masked_sentence(
    sentence=str(example_sentence),
    masked_word_span=parse_span(example_word_span),
    n_candidates=100,
    top_k_labels=5,
    model="Vabamorf",
    ambiguity_policy="split",
)
print("Predicted best label:", example["best_label"])
print("Top label predictions:")
for label_info in example["top_labels"]:
    print(f"  Label: {label_info['label']}, Score: {label_info['score']:.4f}")
print(f"Candidates: {[c['token'] for c in example['candidate_details']]}")

Original sentence: Ma näen rajatise kõrval olevat maja.
Word span to mask: (31, 35)
To be masked word: maja
Masked sentence: Ma näen rajatise kõrval olevat <mask>.
Predicted best label: sg p
Top label predictions:
  Label: sg p, Score: 0.4650
  Label: ?, Score: 0.2350
  Label: sg n, Score: 0.0485
  Label: pl p, Score: 0.0207
  Label: adt, Score: 0.0074
Candidates: ['t', 'tuld', 'inimest', 'm', 'hoonet', 'maja', 'autot', 'elu', 'valgust', 'tähte', 'teed', 'koera', 'varju', 'kassi', 'vett', 'asja', 's', 'last', 'pilti', 'inimesi', 'trolli', 'sõidukit', 'masinat', 'silda', 'sil', 'objekti', 'eset', 'ust', 'S', 'meest', 'looma', 'k', 'puud', 'puid', 'oja', 'L', 'naist', 'lippu', 'O', 'n', 'rongi', 'bussi', 'tanki', 'vihma', 'D', 'T', 'r', 'x', 'korda', 'päikest', 'A', 'tul', 'poega', 'd', 'tn', 'jõge', 'aega', 'suitsu', 'isikut', 'taevast', 'tulekahju', 'rohelist', 'C', 'a', 'j', 'R', 'lapsi', 'olukorda', 'H', 'punast', 'tel', 'kana', 'taime', 'toru', 'ruumi', 'relva', 'N', 'P', 'halli', '

In [19]:
# Run the prediction function on the full dataset and save results.
full_results = []
full_results_candidates = []
n_candidates = 100
for index, row in tqdm.tqdm(
    homonyms_df.iterrows(), total=len(homonyms_df), desc="Predicting labels"
):
    inflection_type = row.get("inflection_type", None)
    sentence = str(row.get("sentence", ""))
    word = str(row.get("word", ""))
    masked_word_span = parse_span(row.get("word_span", (-1, -1)))
    label = parse_label_form(row.get("label", ""))
    source = row.get("source", None)
    result = predict_form_label_from_masked_sentence(
        sentence=sentence,
        masked_word_span=masked_word_span,
        n_candidates=n_candidates,
        top_k_labels=5,
        model="Vabamorf",
        ambiguity_policy="split",
    )
    full_results.append(
        {
            "row_index": index,
            "inflection_type": inflection_type,
            "sentence": sentence,
            "word": word,
            "word_span": masked_word_span,
            "true_label": label,
            "predicted_best_label": result["best_label"],
            "predicted_best_label_score": result["best_label_score"],
            "predicted_top_labels": result["top_labels"],
            "label_scores": result["label_scores"],
            "source": source,
        }
    )
    full_results_candidates.append(
        {
            "row_index": index,
            "inflection_type": inflection_type,
            "sentence": sentence,
            "word": word,
            "word_span": masked_word_span,
            "true_label": label,
            "predicted_best_label": result["best_label"],
            "predicted_best_label_score": result["best_label_score"],
            "predicted_top_labels": result["top_labels"],
            "label_scores": result["label_scores"],
            "candidate_details": result["candidate_details"],
            "source": source,
        }
    )

Predicting labels: 100%|██████████| 7886/7886 [1:49:21<00:00,  1.20it/s]  


In [20]:
# Convert results to DataFrame and save to file.
homonyms_results_df = pd.DataFrame(full_results)
homonyms_results_path = (
    HOMONYMS_DIRS["processed"] / f"homonyms_bert_mlm_predictions_{n_candidates}.parquet"
)
homonyms_results_df.to_parquet(homonyms_results_path, index=False)
# Save the more detailed candidate-level results to a JSONL file for deeper analysis.
homonyms_candidates_jsonl_path = (
    HOMONYMS_DIRS["processed"]
    / f"homonyms_bert_mlm_candidate_details_{n_candidates}.jsonl"
)
with open(homonyms_candidates_jsonl_path, "w", encoding="utf-8") as jsonl_file:
    for item in full_results_candidates:
        jsonl_file.write(json.dumps(item, ensure_ascii=False) + "\n")